In [0]:
#1 How will you process millions of rainfall, traffic, and complaint records?

# Read datasets from Bronze tables

rainfall_df = spark.table("workspace.default.bronze_rainfall")

traffic_df = spark.table("workspace.default.bronze_traffic")

complaints_df = spark.table(
    "workspace.default.bronze_road_complaints"
)

# Check record counts

print("Rainfall records:", rainfall_df.count())
print("Traffic records:", traffic_df.count())
print("Complaint records:", complaints_df.count())

# Example aggregation

from pyspark.sql.functions import avg

rainfall_summary = rainfall_df.groupBy("zone_id").agg(
    avg("rainfall_mm").alias("average_rainfall")
)

display(rainfall_summary)

In [0]:
#2 How will you implement Bronze, Silver, and Gold architecture?

#I will implement Medallion Architecture using Delta Lake in Databricks.
#- Bronze: Raw ingested data.
#- Silver: Cleaned and validated data.
#- Gold: Business-ready aggregated data.

# Bronze layer - read raw ingested data

bronze_rainfall = spark.table("workspace.default.bronze_rainfall")

# Silver layer - read the cleaned Silver table
silver_rainfall = spark.table("workspace.default.silver_rainfall")

# Gold layer - aggregate rainfall by zone

from pyspark.sql.functions import avg, sum, max

gold_rainfall = silver_rainfall.groupBy("zone_id", "zone_name").agg(
    avg("rainfall_mm").alias("average_rainfall"),
    sum("rainfall_mm").alias("total_rainfall"),
    max("rainfall_mm").alias("maximum_rainfall"))

# Save Gold table

gold_rainfall.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_rainfall_summary")

In [0]:
# How will you perform large-scale data cleaning and transformation?

# answer: I will use PySpark DataFrame functions such as dropDuplicates(), filter(), withColumn(), cast() and when() to clean and transform large datasets.

from pyspark.sql.functions import col

# Read rainfall data

rainfall_df = spark.table(
    "workspace.default.bronze_rainfall"
)

# Remove completely duplicate rows

rainfall_clean = rainfall_df.dropDuplicates()

# Filter invalid negative rainfall values

rainfall_clean = rainfall_clean.filter(
    col("rainfall_mm").isNull() |
    (col("rainfall_mm") >= 0)
)

# Check the result

print("Cleaned records:", rainfall_clean.count())

display(rainfall_clean)

In [0]:
# How will you join rainfall, traffic, road damage, and population datasets?

# answer: I will use PySpark joins to combine datasets using common business keys such as zone_id. I will aggregate datasets to a common level before joining to avoid multiplying records.

from pyspark.sql.functions import avg, count, col, when

# Load the road complaints and population datasets
road_df = spark.table("workspace.default.bronze_road_complaints")
population_df = spark.table("workspace.default.bronze_population")

# Rainfall aggregation by zone
rainfall_summary = rainfall_df.groupBy("zone_id").agg(
    avg("rainfall_mm").alias("average_rainfall"))

# Traffic aggregation by zone
# congestion_level is categorical (Low/Medium/High),
# so map it to an ordinal scale before averaging
traffic_summary = traffic_df.groupBy("zone_id").agg(
    avg(when(col("congestion_level") == "Low", 1)
        .when(col("congestion_level") == "Medium", 2)
        .when(col("congestion_level") == "High", 3)
    ).alias("average_congestion"))

# Road complaint aggregation by zone

road_summary = road_df.groupBy("zone_id").agg(
    count("*").alias("total_road_complaints"))

# Select one population record per zone
# Use this only if population has repeated zone records
population_summary = population_df.select(
    "zone_id", "population_density_per_sq_km"
).dropDuplicates(["zone_id"])

# Join all four datasets

combined_df = rainfall_summary \
    .join(traffic_summary, "zone_id", "left") \
    .join(road_summary, "zone_id", "left") \
    .join(population_summary, "zone_id", "left")

display(combined_df)

In [0]:
# How will you identify high-risk flood zones using historical patterns?

#Calculate average and maximum rainfall by zone.
#Count flood incidents by zone.
#Count waterlogging records by zone.
#Consider population density.
# Combine the indicators to create a risk classification.

#Rainfall summary:
from pyspark.sql.functions import avg, max, sum

rainfall_risk = rainfall_df.groupBy(
    "zone_id", "zone_name").agg(
    avg("rainfall_mm").alias("average_rainfall"),
    max("rainfall_mm").alias("maximum_rainfall"),
    sum("rainfall_mm").alias("total_rainfall"))

# Flood incident count:

from pyspark.sql.functions import count

# Load the flood incidents dataset
flood_df = spark.table("workspace.default.bronze_flood")

flood_risk = flood_df.groupBy("zone_id").agg(
    count("*").alias("flood_incident_count"))

# Combine rainfall and flood information:

flood_analysis = rainfall_risk.join(flood_risk,"zone_id","left")

display(flood_analysis)

In [0]:
# How will you implement incremental data processing?

from pyspark.sql.functions import col

last_processed_timestamp = "2026-11-15 00:00:00"

new_records = rainfall_df.filter(
    col("ingestion_timestamp") > last_processed_timestamp
)

display(new_records)

new_records.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.silver_rainfall_incremental")

In [0]:
# How will you optimize large datasets using partitioning?

from pyspark.sql.functions import year, col

# Create year column

rainfall_partitioned = rainfall_df.withColumn(
    "year",
    year(col("observation_date"))
)

# Write partitioned Delta table

rainfall_partitioned.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year") \
    .saveAsTable(
        "workspace.default.silver_rainfall_partitioned"
    )

In [0]:
# How will you create aggregated Gold-layer datasets for reporting?

from pyspark.sql.functions import avg, sum, max, count

gold_rainfall_df = rainfall_df.groupBy(
    "zone_id", "zone_name"
).agg(
    sum("rainfall_mm").alias("total_rainfall"),
    avg("rainfall_mm").alias("average_rainfall"),
    max("rainfall_mm").alias("maximum_rainfall"),
    count("rainfall_mm").alias("valid_rainfall_records")
)

# Save as Gold Delta table
# The table schema changes (valid_rainfall_records added),
# so overwriteSchema is needed to replace the existing schema

gold_rainfall_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.gold_rainfall_summary"
    )

display(gold_rainfall_df)